In [1]:
import pandas as pd

# Load both sheets
df_2009_2010 = pd.read_excel('./data/online_retail_II.xlsx', sheet_name='Year 2009-2010')
df_2010_2011 = pd.read_excel('./data/online_retail_II.xlsx', sheet_name='Year 2010-2011')

# Combine them
df_full = pd.concat([df_2009_2010, df_2010_2011], ignore_index=True)

print(f"New start date: {df_full['InvoiceDate'].min()}")
print(f"New end date: {df_full['InvoiceDate'].max()}")

New start date: 2009-12-01 07:45:00
New end date: 2011-12-09 12:50:00


In [14]:
df_full.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [2]:
df_full.describe()

,Quantity,InvoiceDate,Price,Customer ID
count,1.067371e+06,1067371,1.067371e+06,824364.000000
mean,9.938898e+00,2011-01-02 21:13:55.394028544,4.649388e+00,15324.638504
min,-8.099500e+04,2009-12-01 07:45:00,-5.359436e+04,12346.000000
25%,1.000000e+00,2010-07-09 09:46:00,1.250000e+00,13975.000000
50%,3.000000e+00,2010-12-07 15:28:00,2.100000e+00,15255.000000
75%,1.000000e+01,2011-07-22 10:23:00,4.150000e+00,16797.000000
max,8.099500e+04,2011-12-09 12:50:00,3.897000e+04,18287.000000
std,1.727058e+02,NaN,1.235531e+02,1697.464450


In [3]:
df_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB


In [4]:
df_full['Customer ID'].nunique()

5942

In [5]:
df_full['InvoiceDate']

0         2009-12-01 07:45:00
1         2009-12-01 07:45:00
2         2009-12-01 07:45:00
3         2009-12-01 07:45:00
4         2009-12-01 07:45:00
                  ...        
1067366   2011-12-09 12:50:00
1067367   2011-12-09 12:50:00
1067368   2011-12-09 12:50:00
1067369   2011-12-09 12:50:00
1067370   2011-12-09 12:50:00
Name: InvoiceDate, Length: 1067371, dtype: datetime64[ns]

In [13]:
df_full['Invoice']

0          489434
1          489434
2          489434
3          489434
4          489434
            ...  
1067366    581587
1067367    581587
1067368    581587
1067369    581587
1067370    581587
Name: Invoice, Length: 1067371, dtype: object

In [6]:
df_full['Quantity'].sort_values(ascending=True)

1065883   -80995
587085    -74215
303996     -9600
750990     -9600
750991     -9600
           ...  
127168     12960
127166     12960
90857      19152
587080     74215
1065882    80995
Name: Quantity, Length: 1067371, dtype: int64

In [7]:
df_full['Price'].sort_values(ascending=True)

179403    -53594.36
276274    -44031.79
403472    -38925.87
825444    -11062.06
825445    -11062.06
             ...   
1050063    17836.46
320581     18910.69
241824     25111.09
241827     25111.09
748142     38970.00
Name: Price, Length: 1067371, dtype: float64

In [8]:
df_full['StockCode']

0           85048
1          79323P
2          79323W
3           22041
4           21232
            ...  
1067366     22899
1067367     23254
1067368     23255
1067369     22138
1067370      POST
Name: StockCode, Length: 1067371, dtype: object

In [9]:
df_full['Country']

0          United Kingdom
1          United Kingdom
2          United Kingdom
3          United Kingdom
4          United Kingdom
                ...      
1067366            France
1067367            France
1067368            France
1067369            France
1067370            France
Name: Country, Length: 1067371, dtype: object

In [10]:
from sqlalchemy import create_engine
from sqlalchemy import text 
import pandas as pd

In [11]:
from dotenv import dotenv_values

config = dotenv_values()

pg_user = config['POSTGRES_USER']  
pg_host = config['POSTGRES_HOST']
pg_port = config['POSTGRES_PORT']
pg_db = config['POSTGRES_DB']
pg_schema = config['POSTGRES_SCHEMA']
pg_pass = config['POSTGRES_PASS']

In [12]:
# updating the url
url = f'postgresql://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_db}' #the same like version 1

# let's switch the logging off again. .
engine = create_engine(url, echo=False) #the same like version 1


# writing dataframe to DB : Pandas Dataframe to DB Table in my own Schema
df_full.to_sql(name = 'online_retail_II', 
                       con = engine, 
                       schema = pg_schema, # pandas is allowing to specify, in which schema the table shall be created
                       if_exists='replace', 
                       index=False
                      )

371